# Probabilistic Graphical Models — Exercises

Companion to the note [Probabilistic Graphical Models](Probabilistic%20Graphical%20Models.md).

Practise Bayesian networks (factorization, parameter counting, d-separation, explaining away), Markov random fields (potentials and the partition function), exact inference (enumeration, variable elimination, the HMM forward and Viterbi algorithms), learning CPTs by counting, and Gibbs sampling in an MRF, by hand first and then in NumPy, always checked against brute-force enumeration.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

import itertools
rng = np.random.default_rng(0)
# Shared "sprinkler" network:  Rain -> Wet <- Sprinkler   (all binary, index 1 = true)
P_R = np.array([0.8, 0.2])                # P(R)
P_S = np.array([0.9, 0.1])                # P(S), independent of R
P_W1 = np.array([[0.01, 0.9],             # P(W=1 | R=r, S=s), indexed [r, s]
                 [0.80, 0.99]])

## Part A · Concepts

### Exercise 1 · Chain, fork, collider
*🧠 Concept · ★☆☆*

For the three-node structures (i) chain $A\to B\to C$, (ii) fork $A\leftarrow B\to C$, (iii) collider $A\to B\leftarrow C$, state whether $A\perp C$
holds (a) with nothing observed and (b) given $B$. Give a real-world example of each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Information flows through chains and forks unless the middle node is observed; a collider blocks unless it (or a descendant) is observed.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| | $A\perp C$ | $A\perp C\mid B$ |
|---|---|---|
| chain | no | **yes** |
| fork | no | **yes** |
| collider | **yes** | no |

Chain: smoking → tar in lungs → cancer (given tar, smoking adds nothing). Fork: ice-cream sales ← summer → sunburn (common cause; conditioning on the season removes the correlation). Collider: talent → famous ← luck (talent and luck are independent, but among famous people they become negatively related, Exercise 6 and Exercise 13).

</details>

### Exercise 2 · Explaining away
*🧠 Concept · ★☆☆*

In the sprinkler network the grass is wet. Intuitively, how does learning that the sprinkler was on change your belief that it rained?
Why is this surprising if Rain and Sprinkler are independent a priori? Name a situation in data analysis where conditioning on a collider creates spurious correlation.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Two competing causes for the same effect.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Wet grass raises $P(\text{rain})$. Learning the sprinkler was on **explains** the wet grass, so $P(\text{rain})$ drops back towards its prior (Exercise 6: from $0.67$ to $0.22$). Rain and Sprinkler are marginally independent, but become dependent once their common effect is observed.
In data analysis: selection bias / Berkson's paradox. E.g. among hospitalized patients (hospitalization is caused by disease A or disease B) the two diseases appear negatively correlated; or studying only admitted students makes test score and GPA look negatively correlated.

</details>

### Exercise 3 · Directed or undirected?
*🧠 Concept · ★★☆*

Compare Bayesian networks and Markov random fields: how are the factors defined, is a normalizer needed, and which independence structures can only one of them express?
Which would you use for (a) a medical diagnosis model with causes and symptoms and (b) labelling pixels of an image (neighbouring pixels tend to agree)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

BN factors are conditional distributions $p(x_i\mid\mathrm{pa}(x_i))$; MRF factors $\psi_c\ge0$ are arbitrary.

</details>

<details>
<summary><b>✅ Solution</b></summary>

BN: product of locally normalized CPTs, so $Z=1$ automatically; easy ancestral sampling and parameter learning by counting. MRF: arbitrary non-negative potentials over cliques, so a global $Z=\sum_x\prod_c\psi_c$ is needed and is usually intractable (learning requires it, too).
Expressiveness: only a BN can represent the collider pattern "$A\perp C$ but $A\not\perp C\mid B$" (explaining away); only an MRF can represent a 4-cycle $A-B-C-D-A$ with $A\perp C\mid\{B,D\}$ and $B\perp D\mid\{A,C\}$.
(a) BN: causal, directional knowledge, CPTs elicited from experts. (b) MRF (Ising/Potts, CRF): symmetric "agree with neighbours" relations with no natural direction.

</details>

### Exercise 4 · Spot the graphical model
*🧠 Concept · ★☆☆*

Draw (in words) the graph for each model from the note's example list and state its key conditional independence:
[[Naive Bayes]], a hidden Markov model, a Gaussian mixture model ([[Gaussian Mixture Models and EM]]), probabilistic PCA ([[PCA]]).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Which variables are latent, which are observed, and what are the parents of each observed variable?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Naive Bayes**: class $Y\to X_j$ for every feature. Features are conditionally independent given the class: $X_j\perp X_k\mid Y$.
- **HMM**: hidden chain $z_1\to z_2\to\dots\to z_T$, each $z_t\to x_t$. Markov property $z_{t+1}\perp z_{1:t-1}\mid z_t$, and $x_t\perp$ everything else $\mid z_t$.
- **GMM**: per data point $z_i\to x_i$ with parameters $\pi,\mu,\Sigma$ shared (plate notation). Points are independent given the parameters.
- **PPCA**: continuous latent $z_i\to x_i$, linear-Gaussian: $x_i=Wz_i+\mu+\varepsilon$; coordinates of $x_i$ are independent given $z_i$ (isotropic noise).

</details>

## Part B · By hand

### Exercise 5 · Counting parameters
*✍️ By hand · ★☆☆*

Five binary variables with structure $A\to C$, $B\to C$, $C\to D$, $C\to E$. How many free parameters does the BN have, and how many does an
unrestricted joint distribution over the five variables need? Write out the factorization.

In [ ]:
n_params_bn = None
n_params_full = None

_parents = {'A': [], 'B': [], 'C': ['A', 'B'], 'D': ['C'], 'E': ['C']}
check('BN parameters', n_params_bn, sum(2 ** len(p) for p in _parents.values()))
check('full joint parameters', n_params_full, 2 ** 5 - 1)

<details>
<summary><b>💡 Hint</b></summary>

A binary node with $k$ binary parents needs one number $P(X=1\mid\text{config})$ for each of the $2^k$ parent configurations.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(a,b,c,d,e)=p(a)p(b)p(c\mid a,b)p(d\mid c)p(e\mid c)$. Parameters $1+1+4+2+2=10$ vs $2^5-1=31$ for the full joint.
With $n$ variables and at most $k$ parents each, a BN needs $O(n2^k)$ instead of $O(2^n)$ numbers: this is the whole point of graphical models.

```python
n_params_bn = 10
n_params_full = 31
```

</details>

### Exercise 6 · Explaining away in numbers
*✍️ By hand · ★★☆*

With the shared sprinkler CPTs ($P(R{=}1)=0.2$, $P(S{=}1)=0.1$, $P(W{=}1\mid r,s)$ = 0.01, 0.9, 0.8, 0.99 for $(r,s)=(0,0),(0,1),(1,0),(1,1)$),
compute $P(W{=}1)$, $P(R{=}1\mid W{=}1)$ and $P(R{=}1\mid W{=}1,S{=}1)$ by enumeration.

In [ ]:
p_w = None
p_r_given_w = None
p_r_given_ws = None

_J = P_R[:, None, None] * P_S[None, :, None] * np.stack([1 - P_W1, P_W1], -1)
check('P(W=1)', p_w, _J[:, :, 1].sum())
check('P(R=1 | W=1)', p_r_given_w, _J[1, :, 1].sum() / _J[:, :, 1].sum())
check('P(R=1 | W=1, S=1)', p_r_given_ws, _J[1, 1, 1] / _J[:, 1, 1].sum())

<details>
<summary><b>💡 Hint 1</b></summary>

List the four joint terms $P(r)P(s)P(W{=}1\mid r,s)$: $0.0072,\ 0.072,\ 0.144,\ 0.0198$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$P(R{=}1\mid W{=}1,S{=}1)$ only uses the two terms with $s=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Joint terms with $W=1$: $(0,0)\ 0.8\cdot0.9\cdot0.01=0.0072$; $(0,1)\ 0.8\cdot0.1\cdot0.9=0.072$; $(1,0)\ 0.2\cdot0.9\cdot0.8=0.144$; $(1,1)\ 0.2\cdot0.1\cdot0.99=0.0198$.
$P(W{=}1)=0.243$; $P(R{=}1\mid W{=}1)=0.1638/0.243\approx0.674$; $P(R{=}1\mid W{=}1,S{=}1)=0.0198/(0.0198+0.072)\approx0.216$.
Observing the sprinkler drops the rain probability from 0.67 almost back to the prior 0.2: explaining away.

```python
p_w = 0.243
p_r_given_w = 0.1638 / 0.243
p_r_given_ws = 0.0198 / 0.0918
```

</details>

### Exercise 7 · d-separation queries
*✍️ By hand · ★★☆*

Graph: $A\to C\leftarrow B$, $C\to D$, $A\to E$. Decide for each statement whether it is implied by d-separation (True) or not (False):

1. $A\perp B$  2. $A\perp B\mid D$  3. $E\perp D\mid C$  4. $E\perp B$  5. $E\perp B\mid C$

Store your answers as a list of five booleans in `dsep`. (The check verifies them numerically on a network with random CPTs.)

In [ ]:
dsep = None  # e.g. [True, False, ...]

# numerical test: build a random BN with this graph and test the independences by enumeration
_r = np.random.default_rng(42)
_pA, _pB = _r.uniform(.2, .8), _r.uniform(.2, .8)
_pC, _pD, _pE = _r.uniform(.1, .9, (2, 2)), _r.uniform(.1, .9, 2), _r.uniform(.1, .9, 2)
_J = np.zeros((2,) * 5)
for a, b, c, d, e in itertools.product([0, 1], repeat=5):
    f = lambda p, v: p if v else 1 - p
    _J[a, b, c, d, e] = f(_pA, a) * f(_pB, b) * f(_pC[a, b], c) * f(_pD[c], d) * f(_pE[a], e)
_idx = dict(A=0, B=1, C=2, D=3, E=4)
def _indep(x, y, z=()):
    keep = sorted([_idx[x], _idx[y]] + [_idx[v] for v in z])
    M = _J.sum(axis=tuple(i for i in range(5) if i not in keep))
    M = np.moveaxis(M, [keep.index(_idx[x]), keep.index(_idx[y])], [0, 1]).reshape(2, 2, -1)
    pz = M.sum((0, 1)); pxz = M.sum(1); pyz = M.sum(0)
    return bool(np.allclose(M * pz, pxz[:, None, :] * pyz[None, :, :], atol=1e-12))
_truth = [_indep('A', 'B'), _indep('A', 'B', ['D']), _indep('E', 'D', ['C']), _indep('E', 'B'), _indep('E', 'B', ['C'])]
check('d-separation answers', None if dsep is None else [bool(v) for v in dsep] == _truth, True)

<details>
<summary><b>💡 Hint 1</b></summary>

List every path between the two variables and check each node on it: a chain/fork node blocks if observed; a collider blocks unless it or a descendant is observed.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

In 2 the collider $C$ is unobserved but its descendant $D$ is observed.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **True**: the only path $A\to C\leftarrow B$ has an unobserved collider (and no observed descendant).
2. **False**: observing $D$, a descendant of the collider $C$, opens the path.
3. **True**: the path $E\leftarrow A\to C\to D$ is blocked at $C$ (observed chain node).
4. **True**: $E\leftarrow A\to C\leftarrow B$ is blocked at the unobserved collider $C$.
5. **False**: observing $C$ opens the collider, and the fork at $A$ is unobserved, so $E$ and $B$ become dependent.

```python
dsep = [True, False, True, True, False]
```

</details>

### Exercise 8 · A tiny MRF and its partition function
*✍️ By hand · ★★☆*

Three binary variables in a chain $x_1-x_2-x_3$ with pairwise potentials $\psi(a,b)=2$ if $a=b$ and $1$ otherwise:
$p(x)=\frac1Z\psi(x_1,x_2)\psi(x_2,x_3)$. Compute $Z$ and $P(x_1=x_3)$. Is $x_1\perp x_3\mid x_2$?

In [ ]:
Z_mrf = None
p_x1_eq_x3 = None

_psi = lambda a, b: 2.0 if a == b else 1.0
_cfg = list(itertools.product([0, 1], repeat=3)); _w = np.array([_psi(a, b) * _psi(b, c) for a, b, c in _cfg])
check('Z', Z_mrf, _w.sum())
check('P(x1 = x3)', p_x1_eq_x3, sum(w for (a, b, c), w in zip(_cfg, _w) if a == c) / _w.sum())

<details>
<summary><b>💡 Hint</b></summary>

Classify the 8 configurations by how many of the two edges agree: both (weight 4), one (weight 2), none (weight 1).

</details>

<details>
<summary><b>✅ Solution</b></summary>

Both edges agree: $000,111$ → $2\cdot4=8$. Exactly one agrees: $001,011,100,110$ → $4\cdot2=8$. None: $010,101$ → $2\cdot1=2$. $Z=18$.
$x_1=x_3$ for $000,111,010,101$: $(4+4+1+1)/18=10/18\approx0.556$.
Yes, $x_1\perp x_3\mid x_2$: in an MRF, separation in the graph implies conditional independence ($x_2$ separates them), even though marginally they are positively correlated.

```python
Z_mrf = 18.0
p_x1_eq_x3 = 10 / 18
```

</details>

### Exercise 9 · Variable elimination on a chain
*✍️ By hand · ★☆☆*

Chain $A\to B\to C$ with $P(A{=}1)=0.3$, $P(B{=}1\mid A)=(0.2,\ 0.7)$ for $A=(0,1)$, $P(C{=}1\mid B)=(0.1,\ 0.6)$ for $B=(0,1)$.
Compute $P(C{=}1)$ by eliminating $A$ first, then $B$. Why is this cheaper than summing the full joint?

In [ ]:
p_c1 = None

_pa = np.array([.7, .3]); _pb = np.array([[.8, .2], [.3, .7]]); _pc = np.array([[.9, .1], [.4, .6]])
check('P(C=1) (brute force)', p_c1, sum(_pa[a] * _pb[a, b] * _pc[b, 1] for a in (0, 1) for b in (0, 1)))

<details>
<summary><b>💡 Hint</b></summary>

$P(B{=}1)=\sum_aP(a)P(B{=}1\mid a)$, then $P(C{=}1)=\sum_bP(b)P(C{=}1\mid b)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$P(B{=}1)=0.7\cdot0.2+0.3\cdot0.7=0.35$; $P(C{=}1)=0.65\cdot0.1+0.35\cdot0.6=0.275$.
Eliminating one variable at a time is a sequence of vector–matrix products: for a chain of $n$ variables with $k$ states it costs $O(nk^2)$ instead of $O(k^n)$ for summing the full joint (Exercise 12). In general the cost is exponential in the treewidth.

```python
p_c1 = 0.275
```

</details>

### Exercise 10 · HMM forward recursion by hand
*✍️ By hand · ★★☆*

A 2-state HMM: initial $\pi=(0.6,0.4)$, transitions $A=\begin{pmatrix}0.7&0.3\\0.4&0.6\end{pmatrix}$ (rows = from), emissions
$B=\begin{pmatrix}0.9&0.1\\0.2&0.8\end{pmatrix}$ (rows = state, columns = symbol). Observations $x=(0,1)$.
Compute $\alpha_1$, $\alpha_2$ with $\alpha_1(k)=\pi_kB_{k,x_1}$, $\alpha_t(k)=B_{k,x_t}\sum_j\alpha_{t-1}(j)A_{jk}$, and $p(x)$.

In [ ]:
alpha1 = None
alpha2 = None
p_obs = None

_pi = np.array([.6, .4]); _A = np.array([[.7, .3], [.4, .6]]); _B = np.array([[.9, .1], [.2, .8]])
_brute = sum(_pi[z1] * _B[z1, 0] * _A[z1, z2] * _B[z2, 1] for z1 in (0, 1) for z2 in (0, 1))
check('alpha_1', alpha1, _pi * _B[:, 0])
check('p(x) (sum over all hidden paths)', p_obs, _brute)

<details>
<summary><b>💡 Hint</b></summary>

$\alpha_1=(0.6\cdot0.9,\ 0.4\cdot0.2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\alpha_1=(0.54,\ 0.08)$. $\alpha_1A=(0.54\cdot0.7+0.08\cdot0.4,\ 0.54\cdot0.3+0.08\cdot0.6)=(0.41,\ 0.21)$; times $B_{\cdot,1}=(0.1,0.8)$: $\alpha_2=(0.041,\ 0.168)$.
$p(x)=0.041+0.168=0.209$. The recursion is variable elimination on the HMM chain, eliminating $z_1,z_2,\dots$ in order.

```python
alpha1 = [0.54, 0.08]
alpha2 = [0.041, 0.168]
p_obs = 0.209
```

</details>

## Part C · Code

### Exercise 11 · Inference by enumeration
*💻 Code · ★☆☆*

Build the full joint table `J[r, s, w]` of the sprinkler network as a NumPy array (use broadcasting), and write
`query(J, target, evidence)` that returns $P(\text{target}=1\mid\text{evidence})$ where variables are axis indices (0=R, 1=S, 2=W) and
`evidence` is a dict `{axis: value}`. Reproduce the three numbers of Exercise 6.

In [ ]:
J_sprinkler = None   # shape (2, 2, 2)
def query(J, target, evidence):
    # TODO
    return None

q1 = query(J_sprinkler, 0, {2: 1})
q2 = query(J_sprinkler, 0, {2: 1, 1: 1})

_t = {(r, s, w): P_R[r] * P_S[s] * (P_W1[r, s] if w else 1 - P_W1[r, s]) for r in (0, 1) for s in (0, 1) for w in (0, 1)}
check('joint sums to 1', None if J_sprinkler is None else J_sprinkler.sum(), 1.0)
check('P(R=1 | W=1)', q1, sum(v for k, v in _t.items() if k[0] == 1 and k[2] == 1) / sum(v for k, v in _t.items() if k[2] == 1))
check('P(R=1 | W=1, S=1)', q2, _t[1, 1, 1] / (_t[1, 1, 1] + _t[0, 1, 1]))

<details>
<summary><b>💡 Hint 1</b></summary>

`J = P_R[:, None, None] * P_S[None, :, None] * np.stack([1 - P_W1, P_W1], axis=-1)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Condition by indexing: build a slice with the evidence values fixed, then sum out everything but the target and normalize.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Enumeration is exact but the table has $2^n$ entries, so it only works for a handful of variables; variable elimination (next exercise) exploits the factorization instead.

```python
J_sprinkler = P_R[:, None, None] * P_S[None, :, None] * np.stack([1 - P_W1, P_W1], axis=-1)
def query(J, target, evidence):
    idx = [slice(None)] * J.ndim
    for ax, v in evidence.items():
        idx[ax] = slice(v, v + 1)
    sub = J[tuple(idx)]
    marg = sub.sum(axis=tuple(a for a in range(J.ndim) if a != target))
    return marg[1] / marg.sum()

q1 = query(J_sprinkler, 0, {2: 1})
q2 = query(J_sprinkler, 0, {2: 1, 1: 1})
```

</details>

### Exercise 12 · Variable elimination on a long chain
*💻 Code · ★★☆*

A chain $X_1\to X_2\to\dots\to X_{12}$ of binary variables with random CPTs (below). Compute $P(X_{12})$ by variable elimination
(repeated vector–matrix products) and verify by brute-force enumeration of all $2^{12}$ configurations. Store the VE result in `p_last`.

In [ ]:
r8 = np.random.default_rng(8)
p_first = np.array([0.3, 0.7])
T_chain = r8.dirichlet([1, 1], size=(11, 2))   # T_chain[t, a, b] = P(X_{t+2}=b | X_{t+1}=a)
p_last = None

_tot = np.zeros(2)
for cfg in itertools.product([0, 1], repeat=12):
    p = p_first[cfg[0]]
    for t in range(11):
        p *= T_chain[t, cfg[t], cfg[t + 1]]
    _tot[cfg[-1]] += p
check('P(X_12) vs brute force', p_last, _tot)

<details>
<summary><b>💡 Hint</b></summary>

Start with the vector $P(X_1)$ and repeatedly multiply by the transition matrix of the next step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

VE needs 11 products of a 2-vector with a $2\times2$ matrix (44 multiplications); brute force sums 4096 terms of 12 factors each. For $n=100$ brute force is impossible ($2^{100}$), VE still trivial.

```python
r8 = np.random.default_rng(8)
p_first = np.array([0.3, 0.7])
T_chain = r8.dirichlet([1, 1], size=(11, 2))
p_last = p_first.copy()
for t in range(11):
    p_last = p_last @ T_chain[t]
```

</details>

### Exercise 13 · Explaining away by sampling
*💻 Code · ★★☆*

Draw 200 000 samples from the sprinkler network by **ancestral sampling** (parents before children). Estimate $P(R{=}1\mid W{=}1)$ and
$P(R{=}1\mid W{=}1,S{=}1)$ from the samples (rejection: keep only samples matching the evidence). Also compute the correlation between $R$ and $S$
in all samples and among samples with $W=1$. Store `est_rw`, `est_rws`, `corr_all`, `corr_w`.

In [ ]:
est_rw = est_rws = corr_all = corr_w = None

check('P(R=1 | W=1)', est_rw, 0.1638 / 0.243, tol=0.01)
check('P(R=1 | W=1, S=1)', est_rws, 0.0198 / 0.0918, tol=0.02)
check('independent overall, negatively correlated given W=1', None if corr_all is None else (abs(corr_all) < 0.01 and corr_w < -0.2), True)

<details>
<summary><b>💡 Hint</b></summary>

`R = rng.random(n) < 0.2`; `S = rng.random(n) < 0.1`; `W = rng.random(n) < P_W1[R.astype(int), S.astype(int)]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The estimates match Exercise 6 within sampling error. Overall $\mathrm{corr}(R,S)\approx0$, but among wet-grass samples it is clearly negative ($\approx-0.76$): conditioning on the collider creates a dependence.
Rejection sampling wastes the samples that do not match the evidence; with rare evidence this becomes hopeless, which motivates likelihood weighting and [[MCMC]] (Exercise 17).

```python
_r = np.random.default_rng(12); n = 200000
R = _r.random(n) < P_R[1]
S = _r.random(n) < P_S[1]
W = _r.random(n) < P_W1[R.astype(int), S.astype(int)]
est_rw = R[W].mean()
est_rws = R[W & S].mean()
corr_all = np.corrcoef(R, S)[0, 1]
corr_w = np.corrcoef(R[W], S[W])[0, 1]
print(est_rw, est_rws, corr_all, corr_w)
```

</details>

### Exercise 14 · Learning CPTs by counting
*💻 Code · ★☆☆*

With fully observed data the MLE of each CPT entry is a conditional frequency. Using samples $R,S,W$ (generate 50 000 by ancestral sampling),
estimate the table $\hat P(W{=}1\mid r,s)$ as a $2\times2$ array `P_W1_hat`. Which entry is least accurate and why? How would a Beta prior help?

In [ ]:
P_W1_hat = None

check('estimated CPT', P_W1_hat, P_W1, tol=0.03)

<details>
<summary><b>💡 Hint</b></summary>

`P_W1_hat[r, s] = W[(R == r) & (S == s)].mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The $(r,s)=(1,1)$ entry is estimated from only $\approx2\%$ of the samples ($0.2\cdot0.1$), so it has the largest standard error. With sparse parent configurations some counts can even be zero; a $\mathrm{Beta}(1,1)$ prior (add-one smoothing, [[Bayesian Inference]]) avoids 0/0 and overconfident 0 or 1 estimates. With latent variables counting is impossible and EM uses expected counts instead.

```python
_r = np.random.default_rng(13); n = 50000
R = (_r.random(n) < P_R[1]).astype(int)
S = (_r.random(n) < P_S[1]).astype(int)
W = (_r.random(n) < P_W1[R, S]).astype(int)
P_W1_hat = np.array([[W[(R == r) & (S == s)].mean() for s in (0, 1)] for r in (0, 1)])
print(np.round(P_W1_hat, 3), np.bincount(2 * R + S))
```

</details>

### Exercise 15 · HMM forward algorithm
*💻 Code · ★★☆*

Implement `forward(pi, A, B, obs)` returning $p(x_{1:T})$ for a discrete HMM, scaling $\alpha_t$ at each step (store $\log$ normalizers) to avoid underflow,
so it returns $\log p(x_{1:T})$. Test on the 3-state HMM below with $T=8$ against brute-force summation over all $3^8$ hidden paths.

In [ ]:
pi3 = np.array([0.5, 0.3, 0.2])
A3 = np.array([[0.8, 0.15, 0.05], [0.1, 0.8, 0.1], [0.2, 0.2, 0.6]])
B3 = np.array([[0.7, 0.2, 0.1], [0.1, 0.6, 0.3], [0.3, 0.3, 0.4]])
obs3 = np.array([0, 0, 1, 2, 2, 1, 0, 2])
def forward(pi, A, B, obs):
    # TODO: return log p(obs)
    return None

logp_fwd = forward(pi3, A3, B3, obs3)

_tot = 0.0
for z in itertools.product(range(3), repeat=len(obs3)):
    p = pi3[z[0]] * B3[z[0], obs3[0]]
    for t in range(1, len(obs3)):
        p *= A3[z[t - 1], z[t]] * B3[z[t], obs3[t]]
    _tot += p
check('log p(x) vs brute force', logp_fwd, np.log(_tot))

<details>
<summary><b>💡 Hint</b></summary>

`alpha = pi * B[:, obs[0]]`; then `alpha = (alpha @ A) * B[:, obs[t]]`; normalize `c = alpha.sum()`, `alpha /= c`, accumulate `log c`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Cost $O(TK^2)$ instead of $O(K^T)$. Without scaling, $\alpha_t$ shrinks geometrically and underflows for sequences of a few hundred steps; the sum of the log normalizers is exactly $\log p(x_{1:T})$.
The normalized $\alpha_t$ is the filtering distribution $p(z_t\mid x_{1:t})$; the Kalman filter is the same recursion for linear-Gaussian models.

```python
pi3 = np.array([0.5, 0.3, 0.2])
A3 = np.array([[0.8, 0.15, 0.05], [0.1, 0.8, 0.1], [0.2, 0.2, 0.6]])
B3 = np.array([[0.7, 0.2, 0.1], [0.1, 0.6, 0.3], [0.3, 0.3, 0.4]])
obs3 = np.array([0, 0, 1, 2, 2, 1, 0, 2])
def forward(pi, A, B, obs):
    alpha = pi * B[:, obs[0]]
    logp = 0.0
    for t in range(len(obs)):
        if t > 0:
            alpha = (alpha @ A) * B[:, obs[t]]
        c = alpha.sum()
        alpha = alpha / c
        logp += np.log(c)
    return logp

logp_fwd = forward(pi3, A3, B3, obs3)
```

</details>

### Exercise 16 · Viterbi decoding
*💻 Code · ★★★*

Implement `viterbi(pi, A, B, obs)` returning the most probable hidden path $\arg\max_zp(z,x)$ in log space (max-product instead of sum-product,
with back-pointers). Verify on the HMM of the previous exercise against the brute-force argmax over all $3^8$ paths.

In [ ]:
def viterbi(pi, A, B, obs):
    # TODO: return a list/array of T states
    return None

path_vit = viterbi(pi3, A3, B3, obs3)

_best, _bp = -np.inf, None
for z in itertools.product(range(3), repeat=len(obs3)):
    lp = np.log(pi3[z[0]]) + np.log(B3[z[0], obs3[0]]) + sum(np.log(A3[z[t - 1], z[t]]) + np.log(B3[z[t], obs3[t]]) for t in range(1, len(obs3)))
    if lp > _best:
        _best, _bp = lp, z
check('Viterbi path equals brute-force argmax', None if path_vit is None else list(map(int, path_vit)) == list(_bp), True)

<details>
<summary><b>💡 Hint 1</b></summary>

$\delta_t(k)=\log B_{k,x_t}+\max_j[\delta_{t-1}(j)+\log A_{jk}]$; store the argmax $j$ as back-pointer $\psi_t(k)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Backtrack from $\arg\max_k\delta_T(k)$ using $\psi$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Viterbi is the same dynamic programme as the forward algorithm with $\sum$ replaced by $\max$ (max-product belief propagation). Note that the jointly most probable path can differ from the sequence of individually most probable states $\arg\max_kp(z_t=k\mid x)$ from forward–backward.

```python
def viterbi(pi, A, B, obs):
    T, K = len(obs), len(pi)
    lA, lB = np.log(A), np.log(B)
    delta = np.log(pi) + lB[:, obs[0]]
    psi = np.zeros((T, K), dtype=int)
    for t in range(1, T):
        scores = delta[:, None] + lA          # [from j, to k]
        psi[t] = scores.argmax(0)
        delta = scores.max(0) + lB[:, obs[t]]
    path = [int(delta.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(psi[t, path[-1]]))
    return path[::-1]

path_vit = viterbi(pi3, A3, B3, obs3)
print(path_vit)
```

</details>

### Exercise 17 · Gibbs sampling in an Ising MRF
*💻 Code · ★★★*

A $3\times3$ Ising model with spins $x_i\in\{-1,+1\}$: $p(x)\propto\exp\big(J\sum_{(i,j)\in E}x_ix_j+\sum_ih_ix_i\big)$ with grid edges, $J=0.4$ and field
$h_i=0.2$ on the top-left spin only, 0 elsewhere. Run a Gibbs sampler (each spin from $p(x_i=+1\mid x_{-i})=\sigma\big(2(J\sum_{j\in N(i)}x_j+h_i)\big)$) for 20 000 sweeps
and estimate $P(x_i=+1)$ for all 9 spins (`gibbs_marg`, shape (3,3)). The check computes the exact marginals by enumerating all 512 states (so $Z$ is computed exactly).

In [ ]:
J_is = 0.4
h_is = np.zeros((3, 3)); h_is[0, 0] = 0.2
gibbs_marg = None

_cfgs = np.array(list(itertools.product([-1, 1], repeat=9))).reshape(-1, 3, 3)
_E = J_is * ((_cfgs[:, :, :-1] * _cfgs[:, :, 1:]).sum((1, 2)) + (_cfgs[:, :-1, :] * _cfgs[:, 1:, :]).sum((1, 2))) + (_cfgs * h_is).sum((1, 2))
_w = np.exp(_E); _w /= _w.sum()
_exact = np.tensordot(_w, (_cfgs == 1).astype(float), axes=1)
check('marginals P(x_i = +1)', gibbs_marg, _exact, tol=0.02)

<details>
<summary><b>💡 Hint 1</b></summary>

Neighbour sum with zero padding: `np.pad(x, 1)` then add the four shifted views; or loop over the 4 neighbours with bounds checks.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Update spins one at a time (systematic scan), record after each full sweep, discard the first 1 000 sweeps.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The field on one corner spin propagates through the couplings: that spin has $P(+1)\approx0.60$ and the effect decays with graph distance (the opposite corner is only slightly above 0.5).
For a $3\times3$ grid we can enumerate $2^9$ states, but for a $100\times100$ image $Z$ has $2^{10000}$ terms: Gibbs sampling ([[MCMC]]), loopy BP or mean-field [[Variational Inference]] are the only options. Near the critical coupling ($J\approx0.44$ on large grids) Gibbs mixes very slowly.

```python
J_is = 0.4
h_is = np.zeros((3, 3)); h_is[0, 0] = 0.2
_r = np.random.default_rng(14)
x = _r.choice([-1, 1], size=(3, 3))
acc = np.zeros((3, 3)); n_keep = 0
for sweep in range(20000):
    for i in range(3):
        for j in range(3):
            s = sum(x[a, b] for a, b in ((i - 1, j), (i + 1, j), (i, j - 1), (i, j + 1)) if 0 <= a < 3 and 0 <= b < 3)
            p_plus = 1 / (1 + np.exp(-2 * (J_is * s + h_is[i, j])))
            x[i, j] = 1 if _r.random() < p_plus else -1
    if sweep >= 1000:
        acc += (x == 1); n_keep += 1
gibbs_marg = acc / n_keep
print(np.round(gibbs_marg, 3))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Probabilistic Graphical Models](Probabilistic%20Graphical%20Models.md).*